In [2]:
# -*- coding: utf-8 -*-


normalization.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1JFP5yaFpvClqWuXhF73mDezFDzOzVBBd


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip install indic-transliteration --quiet
!pip install rapidfuzz --quiet

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import re
import json
import unicodedata
import pandas as pd
from google.colab import files, data_table

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate as sanscript_transliterate

print("✓ All imports OK")

✓ All imports OK


In [5]:
BASE = '/content/drive/MyDrive/AmazonMLChallenge/student_resource/dataset'

training_s1  = pd.read_csv(f'{BASE}/train/train_source1.tsv', sep='\t', dtype=str)
training_s2  = pd.read_csv(f'{BASE}/train/train_source2.tsv', sep='\t', dtype=str)
training_s3  = pd.read_csv(f'{BASE}/train/train_source3.tsv', sep='\t', dtype=str)
ground_truth = pd.read_csv(f'{BASE}/train/train_ground_truth.tsv', sep='\t', dtype=str)
test_s1      = pd.read_csv(f'{BASE}/test/test_source1.tsv', sep='\t', dtype=str)
test_s2      = pd.read_csv(f'{BASE}/test/test_source2.tsv', sep='\t', dtype=str)
test_s3      = pd.read_csv(f'{BASE}/test/test_source3.tsv', sep='\t', dtype=str)

print(f"Train  — S1: {len(training_s1):,}  S2: {len(training_s2):,}  S3: {len(training_s3):,}")
print(f"GT     — {len(ground_truth):,} rows")
print(f"Test   — S1: {len(test_s1):,}  S2: {len(test_s2):,}  S3: {len(test_s3):,}")

Train  — S1: 2,206,821  S2: 5,034,616  S3: 5,285,603
GT     — 2,206,821 rows
Test   — S1: 1,732,544  S2: 4,887,273  S3: 5,082,316


In [6]:
null_rates = {}
for i, df in enumerate([training_s1, training_s2, training_s3], start=1):
    null_rates[f'training_s{i}'] = df.isna().sum().to_dict()

null_rates['singleton_fraction'] = (
    ground_truth['matched_entity_ids'].replace('', float('nan')).isna().mean()
)

with open('null_rates.json', 'w') as f:
    json.dump(null_rates, f, indent=4)

print(json.dumps(null_rates, indent=2))
files.download('null_rates.json')

{
  "training_s1": {
    "entity_id": 0,
    "business_name": 0,
    "business_address": 0,
    "country": 0
  },
  "training_s2": {
    "entity_id": 0,
    "business_name": 2,
    "business_address": 168967,
    "country": 0
  },
  "training_s3": {
    "entity_id": 0,
    "business_name": 13,
    "business_address": 175916,
    "country": 0
  },
  "singleton_fraction": 0.05584820880352326
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
SCRIPT_PATTERNS = {
    'devanagari': r'[\u0900-\u097F]',
    'bengali':    r'[\u0980-\u09FF]',
    'tamil':      r'[\u0B80-\u0BFF]',
    'telugu':     r'[\u0C00-\u0C7F]',
    'gujarati':   r'[\u0A80-\u0AFF]',
    'gurmukhi':   r'[\u0A00-\u0A7F]',
    'kannada':    r'[\u0C80-\u0CFF]',
    'malayalam':  r'[\u0D00-\u0D7F]',
}

def detect_script(text):
    if not isinstance(text, str):
        return 'latin'
    counts = {}
    for script_name, pattern in SCRIPT_PATTERNS.items():
        matches = len(re.findall(pattern, text))
        if matches > 0:
            counts[script_name] = matches
    return max(counts, key=counts.get) if counts else 'latin'

# Quick sanity check
assert detect_script("ब्लू टेक्नोलॉजीज") == "devanagari"
assert detect_script("Acme Corp") == "latin"
print("✓ detect_script OK")

✓ detect_script OK


In [8]:
TARGET_SCHEME = sanscript.ITRANS

SANSCRIPT_SOURCE_SCHEME = {
    'devanagari': sanscript.DEVANAGARI,
    'bengali':    sanscript.BENGALI,
    'gurmukhi':   sanscript.GURMUKHI,
    'gujarati':   sanscript.GUJARATI,
    'tamil':      sanscript.TAMIL,
    'telugu':     sanscript.TELUGU,
    'kannada':    sanscript.KANNADA,
    'malayalam':  sanscript.MALAYALAM,
}

def transliterate_to_latin(text, script):
    """
    Transliterate non-Latin → Latin (ITRANS).
    Returns original text unchanged if script is latin or on any failure.
    This runs as the FIRST step in normalize_record().
    """
    if script == 'latin':
        return text
    source_scheme = SANSCRIPT_SOURCE_SCHEME.get(script)
    if source_scheme is None:
        return text
    try:
        return sanscript_transliterate(text, source_scheme, TARGET_SCHEME)
    except Exception:
        return text   # always prefer original over a crash

# Your original test cases
print(transliterate_to_latin("அரிஹந்த்",        "tamil"))       # → arihant
print(transliterate_to_latin("ब्लू टेक्नोलॉजीज", "devanagari"))  # → blue technologies

arihandh
blU TeknolaॉjIja


In [9]:
# ── CELL 7: Lookup tables  (merged: your alias_dict + standard forms) ─────────

# Legal suffix normalization
# All variants → canonical expanded form.
# Your EDA discoveries (limirrad, elaelapi etc.) are preserved here.
LEGAL_SUFFIXES = {
    # ── Multi-word phrases first (longer match wins) ──────────────────────────
    'private limited':  'pvt ltd',
    'pvt ltd':          'pvt ltd',
    'p ltd':            'pvt ltd',

    # ── limited → ltd ─────────────────────────────────────────────────────────
    'limited':          'ltd',
    'ltd':              'ltd',
    'limiteda':         'ltd',        'limirrad':   'ltd',
    'limidhèdh':        'ltd',        'limitèd':    'ltd',
    'límited':          'ltd',        'li':         'ltd',

    # ── private → pvt ─────────────────────────────────────────────────────────
    'private':          'pvt',        'pvt':        'pvt',

    # ── incorporated → inc ────────────────────────────────────────────────────
    'incorporated':     'inc',        'inc':        'inc',
    'ínc':              'inc',

    # ── corporation / company → co ────────────────────────────────────────────
    'corporation':      'co',         'corp':       'co',
    'company':          'co',         'com':        'co',
    'co':               'co',         'c':          'co',

    # ── others ────────────────────────────────────────────────────────────────
    'llc':              'llc',
    'llp':              'llp',        'elaelapi':   'llp',
    'plc':              'plc',
    'center':           'center',     'cénter':     'center',
}

_LEGAL_SUFFIXES_SORTED = sorted(LEGAL_SUFFIXES.keys(), key=len, reverse=True)
print("✓ Lookup tables loaded")

# Street / road type abbreviations  — abbreviated form is canonical
# Long form → short form. Short forms not in this dict pass through unchanged.
STREET_ABBREVS = {
    # English road types
    'road':         'rd',     'street':       'st',
    'avenue':       'ave',    'av':           'ave',   # av = alt abbrev → same canonical
    'boulevard':    'blvd',   'bd':           'blvd',  # bd = French abbrev → same canonical
    'drive':        'dr',     'lane':         'ln',
    'court':        'ct',     'highway':      'hwy',
    'freeway':      'fwy',    'parkway':      'pkwy',
    'place':        'pl',     'square':       'sq',
    'trail':        'trl',    'terrace':      'ter',
    'circle':       'cir',    'expressway':   'expy',
    # Compass directions
    'north':        'n',      'south':        's',
    'east':         'e',      'west':         'w',
    'northeast':    'ne',     'northwest':    'nw',
    'southeast':    'se',     'southwest':    'sw',
    # Building / unit
    'apartment':    'apt',    'suite':        'ste',
    'floor':        'fl',     'building':     'bldg',
    'department':   'dept',
    # Indian address terms
    'near':         'nr',     'opposite':     'opp',
    'adjacent':     'adj',    'junction':     'jn',
    'station':      'stn',    'nagar':        'ng',
    'ngr':          'ng',     'colony':       'col',
    'extension':    'ext',    'society':      'soc',
    'sector':       'sec',    'phase':        'ph',
    'block':        'blk',    'market':       'mkt',
    'compound':     'cmpd',
}

# City name variants: colonial / alternate romanisation → canonical
CITY_VARIANTS = {
    'bangalore': 'bengaluru',  'bombay': 'mumbai',
    'madras': 'chennai',       'calcutta': 'kolkata',
    'poona': 'pune',           'gurgaon': 'gurugram',
    'trivandrum': 'thiruvananthapuram',
    'pondicherry': 'puducherry',
    'baroda': 'vadodara',      'mysore': 'mysuru',
    'mangalore': 'mangaluru',  'hubli': 'hubballi',
    'tumkur': 'tumakuru',      'shimoga': 'shivamogga',
    'belgaum': 'belagavi',     'gulbarga': 'kalaburagi',
    'bijapur': 'vijayapura',   'new delhi': 'delhi',
    'dilli': 'delhi',
}

# PIN / ZIP patterns
_PIN_PATTERNS = [
    re.compile(r'\b([1-9]\d{5})\b'),        # India: 6-digit PIN
    re.compile(r'\b(\d{5})(?:-\d{4})?\b'),  # US / France: 5-digit ZIP
]

✓ Lookup tables loaded


In [10]:
def extract_pin_zip(text):
    """
    Extract PIN/ZIP already present in address text.
    Returns None if not found — never imputes or guesses.

    How to use in blocking:
      Only use as a blocking key when BOTH records return non-None.
      If either is None, fall back to name/token blocking for that pair.
    """
    for pattern in _PIN_PATTERNS:
        m = pattern.search(text)
        if m:
            return m.group(1)
    return None

def extract_legal_suffix(name):
    """Return the canonical legal suffix found in a name, or None."""
    lower = unicodedata.normalize('NFKD', str(name)).encode('ascii', errors='ignore').decode().lower().strip()
    for raw in _LEGAL_SUFFIXES_SORTED:
        if lower.endswith(' ' + raw) or lower == raw:
            return LEGAL_SUFFIXES[raw]
        if re.search(r'\b' + re.escape(raw) + r'\b', lower):
            return LEGAL_SUFFIXES[raw]
    return None

print("✓ Extraction functions ready")

✓ Extraction functions ready


In [11]:
def to_ascii_lower(text):
    nfkd = unicodedata.normalize('NFKD', str(text))
    return nfkd.encode('ascii', errors='ignore').decode('ascii').lower().strip()

def clean_text(text):
    """Your clean_text_column() adapted for strings."""
    text = text.lower()
    text = re.sub(r'&', ' and ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return [t for t in text.split() if t]

def normalize_suffixes(text):
    """
    Your normalize_suffixes() — FIXED: applied to ALL tokens, not just last.
    Uses merged LEGAL_SUFFIXES (your alias_dict + standard forms).
    """
    if not isinstance(text, str) or not text:
        return text
    return ' '.join(LEGAL_SUFFIXES.get(tok, tok) for tok in text.split())

def remove_legal_suffix_from_name(name):
    """Strip legal suffix to get the 'core' name (for name_core feature)."""
    lower = to_ascii_lower(name)
    for raw in _LEGAL_SUFFIXES_SORTED:
        if lower.endswith(' ' + raw):
            return name[:-(len(raw)+1)].strip(' ,.')
        if lower == raw:
            return ''
    return name

def normalize_name(text):
    """
    Full name pipeline:
    ascii_lower → clean_text → normalize_suffixes → expand_street_abbrevs
    """
    text = to_ascii_lower(text)
    text = clean_text(text)
    text = normalize_suffixes(text)
    tokens = tokenize(text)
    tokens = [STREET_ABBREVS.get(t, t) for t in tokens]
    return ' '.join(tokens)

def normalize_address(text):
    """
    Full address pipeline:
    remove_landmark → number_formats → ascii_lower → clean_text
    → expand_abbrevs → normalize_city_variants
    """
    text = to_ascii_lower(text)
    text = clean_text(text)
    tokens = tokenize(text)
    tokens = [STREET_ABBREVS.get(t, t) for t in tokens]
    # Two-token city variants (e.g. 'new delhi' → 'delhi')
    result, i = [], 0
    while i < len(tokens):
        if i + 1 < len(tokens):
            bigram = tokens[i] + ' ' + tokens[i+1]
            if bigram in CITY_VARIANTS:
                result.append(CITY_VARIANTS[bigram])
                i += 2
                continue
        result.append(CITY_VARIANTS.get(tokens[i], tokens[i]))
        i += 1
    return ' '.join(result)

print("✓ Normalization functions ready")

✓ Normalization functions ready


In [3]:
def _safe_str(val):
    """Convert field value to string. Returns '' for None/NaN instead of 'nan'."""
    try:
        if pd.isna(val):
            return ''
    except (TypeError, ValueError):
        pass
    return str(val).strip()

def normalize_record(row):
    """
    Full pipeline for one record (dict or Series with entity_id,
    business_name, business_address, country).

    Pipeline order:
      1. detect_script() + transliterate_to_latin()  ← your code
      2. extract_pin_zip(), extract_landmark()        ← new
      3. normalize_name(), normalize_address()        ← merged

    Output columns used downstream:
      block.py    : country, pin_zip, has_pin, name_tokens, addr_tokens
      features.py : name_norm, name_core, legal_suffix, addr_norm, landmark
    """
    raw_name = str(row.get('business_name', '') or '')
    raw_addr = str(row.get('business_address', '') or '')

    # 1. Script detection + transliteration
    name_script   = detect_script(raw_name)
    addr_script   = detect_script(raw_addr)
    name_translit = transliterate_to_latin(raw_name, name_script)
    addr_translit = transliterate_to_latin(raw_addr, addr_script)

    # 2. Extract before cleaning (PIN regex needs raw digits intact)
    pin       = extract_pin_zip(addr_translit)
    legal_sfx = extract_legal_suffix(name_translit)

    # 3. Normalize
    name_norm = normalize_name(name_translit)
    addr_norm = normalize_address(addr_translit)
    core_name = normalize_name(remove_legal_suffix_from_name(name_translit))

    return {
        'entity_id':    row['entity_id'],
        'country':      to_ascii_lower(str(row.get('country', '') or '')),
        # name
        'name_norm':    name_norm,
        'name_core':    core_name,
        'name_tokens':  sorted(set(tokenize(name_norm))),
        'legal_suffix': legal_sfx,
        'name_script':  name_script,
        # address
        'addr_norm':    addr_norm,
        'addr_tokens':  sorted(set(tokenize(addr_norm))),
        'pin_zip':      pin,
        'has_pin':      pin is not None,
    }

def normalize_dataframe(df):
    """Apply normalize_record to a full source DataFrame."""
    return pd.DataFrame([normalize_record(r) for r in df.to_dict('records')])

print("✓ normalize_record() ready")

✓ normalize_record() ready


In [ ]:
print("Normalizing train sources...")
norm_s1 = normalize_dataframe(training_s1)
print(f"  ✓ S1: {len(norm_s1):,} records")

norm_s2 = normalize_dataframe(training_s2)
print(f"  ✓ S2: {len(norm_s2):,} records")

norm_s3 = normalize_dataframe(training_s3)
print(f"  ✓ S3: {len(norm_s3):,} records")

print("\nNormalizing test sources...")
norm_test_s1 = normalize_dataframe(test_s1)
norm_test_s2 = normalize_dataframe(test_s2)
norm_test_s3 = normalize_dataframe(test_s3)
print(f"  ✓ Test S1: {len(norm_test_s1):,}  S2: {len(norm_test_s2):,}  S3: {len(norm_test_s3):,}")

norm_s1.head(5)

Normalizing train sources...
  ✓ S1: 2,206,821 records


In [ ]:
from rapidfuzz.distance import Jaccard

all_s2_s3     = pd.concat([training_s2, training_s3]).set_index('entity_id')
all_norm_s2_s3 = pd.concat([norm_s2, norm_s3]).set_index('entity_id')
norm_s1_idx    = norm_s1.set_index('entity_id')
train_s1_idx   = training_s1.set_index('entity_id')

sample = ground_truth[ground_truth['matched_entity_ids'] != ''].sample(min(200, len(ground_truth)), random_state=42)

raw_scores, norm_scores = [], []

for _, row in sample.iterrows():
    s1_id   = row['source1_entity_id']
    cand_id = row['matched_entity_ids'].split(',')[0].strip()

    if s1_id not in norm_s1_idx.index or cand_id not in all_norm_s2_s3.index:
        continue

    raw_a  = str(train_s1_idx.loc[s1_id, 'business_name']).lower().split()
    raw_b  = str(all_s2_s3.loc[cand_id, 'business_name']).lower().split()
    norm_a = norm_s1_idx.loc[s1_id, 'name_tokens']
    norm_b = all_norm_s2_s3.loc[cand_id, 'name_tokens']

    raw_scores.append(Jaccard.normalized_similarity(raw_a, raw_b))
    norm_scores.append(Jaccard.normalized_similarity(norm_a, norm_b))

if raw_scores:
    mean_raw  = sum(raw_scores)  / len(raw_scores)
    mean_norm = sum(norm_scores) / len(norm_scores)
    delta     = mean_norm - mean_raw
    status    = '✅ PASS' if delta >= 0.20 else '❌ FAIL — check normalization'
    print(f"{status}")
    print(f"  Mean Jaccard (raw):  {mean_raw:.3f}")
    print(f"  Mean Jaccard (norm): {mean_norm:.3f}")
    print(f"  Improvement:         {delta:+.3f}  (gate: ≥ +0.20)")
else:
    print("No matching pairs found — check entity_id alignment")